# CorpusSLR: a systematic review from one query

This notebook runs a complete, PRISMA-documented literature search.
You do not have to edit any code: run the cells from top to bottom
and answer the questions they ask.

**What you will have at the end**

| File | What it is for |
|---|---|
| `corpus_scopus.csv` | the deduplicated corpus in Scopus column format -- opens in bibliometrix, VOSviewer and EmbedSLR without conversion |
| `screening.csv` | titles and abstracts for ASReview, Rayyan or a spreadsheet |
| `prisma_s_appendix.md` | the search-reporting appendix (PRISMA-S) |
| `prisma2020_flow.svg` | the PRISMA 2020 flow diagram |
| `dedup_report.csv` | every duplicate decision and the rule that made it |
| `review.json` | the search strategy as data -- deposit this and anyone can rerun your search |

**About API keys.** Scopus and Web of Science need a key from your
library. This notebook asks for keys with `getpass`, which does not
echo what you type and does not store it in the notebook file. Never
type a key directly into a cell: a saved notebook keeps cell text and
cell outputs, and both end up in your Drive and in anything you
share. Everything else here works with no key at all, so you can run
a real review without any subscription.

## 1. Install CorpusSLR

CorpusSLR depends only on `requests`, which Colab already has, so
this is quick and cannot break the rest of your environment.
Run the cell and wait for `installed:` to appear. The cell tries the
package index first and the source repository second; if neither is
reachable it says so and tells you how to install from a local copy,
instead of failing later with a bare import error.

In [ ]:
# Step 1: install the package.
#
# Two sources are tried, because which one works depends on where the
# release currently lives: the package index first, then the source
# repository. A bare 'pip install corpusslr' followed by an import
# fails with ModuleNotFoundError when the index copy is not there yet,
# which tells the reader nothing about what to do next.
import subprocess
import sys

SOURCES = ['corpusslr',
           'git+https://github.com/s-matysik/CorpusSLR.git']


def install(spec):
    done = subprocess.run([sys.executable, '-m', 'pip', 'install',
                           '--quiet', spec],
                          capture_output=True, text=True)
    return done.returncode == 0


if not any(install(s) for s in SOURCES):
    raise SystemExit(
        'Could not install corpusslr from the package index or from the '
        'repository.\n'
        'If you have the source tree (for example unzipped in Drive), '
        'install from that copy instead, by running '
        'pip install /content/corpusslr in a new cell.\n'
        'Otherwise use the repository or archive address given in the '
        'code metadata table of the accompanying article.')

import corpusslr
print('installed:', corpusslr.__version__)

## 2. Say who you are (an email address)

Crossref, PubMed and OpenAlex ask automated clients to identify
themselves with a contact address, and they throttle anonymous
clients harder. This is politeness, not authentication: the address
is sent in a request header and is not a credential. It is kept in
this session only.

In [ ]:
# Step 2: contact address for the polite request header.
import os

email = input('Your email address (Enter to skip): ').strip()
if email:
    os.environ['CORPUSSLR_CONTACT_EMAIL'] = email
    print('Will identify requests as:', email)
else:
    print('No address set. The free databases still work, just with '
          'a lower rate limit.')

## 3. API keys, typed hidden

Run this cell only if you have a Scopus or Web of Science key. Press
Enter to skip either one -- skipping is normal and nothing will
break.

`getpass` hides what you type, so the key never appears on screen,
never in the notebook's saved text, and never in a cell output. The
value lives in this session's memory and disappears when the runtime
is recycled.

*Off campus?* An Elsevier key is tied to your institution's IP range;
from outside it also needs an institutional token (`SCOPUS_INSTTOKEN`
below). Ask your library for both.

In [ ]:
# Step 3: API keys. Nothing typed here is echoed or saved.
import getpass
import os

SLOTS = [
    ('SCOPUS_API_KEY', 'Scopus API key'),
    ('SCOPUS_INSTTOKEN', 'Scopus institutional token (off campus)'),
    ('WOS_API_KEY', 'Web of Science API key'),
]

for variable, label in SLOTS:
    value = getpass.getpass('{} (Enter to skip): '.format(label))
    value = (value or '').strip()
    if value:
        os.environ[variable] = value

# Presence only -- the values themselves are never printed.
for variable, label in SLOTS:
    state = 'set' if os.environ.get(variable) else 'not set'
    print('{:<20} {:<9} {}'.format(variable, state, label))

## 4. Choose the databases

No single database covers a field. PRISMA-S (Rethlefsen et al.,
2021) expects a review to name every database it searched, and
Gusenbauer & Haddaway (2020) distinguish *principal* search systems,
which can carry a review on their own (Scopus, Web of Science,
PubMed), from *supplementary* ones that should be added on top rather
than used instead.

The cell lists what is available and marks which need a key. Type the
numbers you want, or the word `free` for everything that needs no
key.

In [ ]:
# Step 4: pick databases.
from corpusslr.cli import DATABASES, credential_status

aliases = sorted(DATABASES)
for i, alias in enumerate(aliases, 1):
    status = credential_status(alias)
    if not status['requires_credential']:
        note = 'free'
    elif status['credential_present']:
        note = 'key is set'
    else:
        note = 'needs a key (step 3) -- would be skipped'
    print('{:>2}) {:<34} {}'.format(i, DATABASES[alias]['label'],
                                   note))

answer = input('\nNumbers separated by commas, or "free": ').strip()

if answer.lower() in ('free', 'keyless', ''):
    databases = [a for a in aliases if DATABASES[a]['keyless']]
else:
    databases = []
    for token in answer.replace(';', ',').split(','):
        token = token.strip()
        if not token:
            continue
        if token.isdigit() and 1 <= int(token) <= len(aliases):
            pick = aliases[int(token) - 1]
        else:
            from corpusslr.cli import _norm_db
            pick = _norm_db(token)
        if not pick:
            print('I do not know a database called {!r}; ignoring it.'
                  .format(token))
        elif pick not in databases:
            databases.append(pick)

if not databases:
    databases = [a for a in aliases if DATABASES[a]['keyless']]
    print('\nNothing recognised, so I selected the free databases.')

print('\nSearching:', ', '.join(DATABASES[a]['label']
                                for a in databases))

## 5. Write the query

A CorpusSLR query is a set of **blocks**. Inside a block the terms are
alternatives joined with OR; the blocks are then joined with AND.
This is the three-block pattern reviewers are expected to report --
what the technology is, what it does, where it is applied:

```
block 1: artificial intelligence, machine learning
block 2: adoption, acceptance
block 3: small business, SME
```

means `(artificial intelligence OR machine learning) AND (adoption OR
acceptance) AND (small business OR SME)`.

Writing the query once and letting CorpusSLR translate it into each
database's own syntax removes the most common methodological problem
in multi-database searching: a query that quietly means something
different in every interface.

Enter one block per line and press Enter on an empty line to finish.

In [ ]:
# Step 5: the query.
blocks = []
print('One block per line, terms separated by commas. '
      'Empty line = done.')
while True:
    line = input('  block {}: '.format(len(blocks) + 1)).strip()
    if not line:
        break
    terms = [t.strip() for t in line.split(',') if t.strip()]
    if any(len(t) < 2 for t in terms):
        print('    A one-character term matches almost everything. '
              'Retype this block.')
        continue
    if terms:
        blocks.append(terms)

years = input('Years as "from-to" (Enter for no limit): ').strip()
year_range = None
if years:
    parts = [p for p in years.replace('..', '-').split('-') if p]
    try:
        year_range = [int(parts[0]), int(parts[-1])]
    except (ValueError, IndexError):
        print('Could not read {!r} as a year range; no year limit set.'
              .format(years))

types = input('Document types (article, review, conference; '
              'Enter for all): ').strip()
doc_types = [t.strip().lower() for t in types.split(',') if t.strip()]

if not blocks:
    print('\nNo query entered. Run this cell again -- the search '
          'cannot proceed without one.')
else:
    print('\n{} block(s):'.format(len(blocks)))
    for i, block in enumerate(blocks, 1):
        print('  {}: {}'.format(i, ' OR '.join(block)))

## 6. Build the search strategy file

Everything you have answered now becomes one JSON file. This file is
the reproducible artefact of your review: it holds the query, the
database list, the year window and every deduplication parameter, and
it holds **no credentials** -- keys are read from the environment, by
design, so the file is safe to deposit as supplementary material.

Anyone with the file can rerun your entire search with one command:
`corpusslr run -c review.json`. That is what makes the search
*repeatable* in the PRISMA-S sense (Item 8) rather than merely
described.

In [ ]:
# Step 6: write review.json.
import json
import os
from corpusslr.cli import validate_config

OUT_DIR = 'corpusslr_review'
os.makedirs(OUT_DIR, exist_ok=True)

config = {
    'review': {'title': 'Review run in Google Colab'},
    'query': {'blocks': blocks},
    'databases': [{'name': a} for a in databases],
    'dedup': {},
    'enrich': {'recover_abstracts': True},
    'output': {'dir': OUT_DIR,
               'exports': ['csv', 'screening', 'ris', 'bibtex'],
               'svg': True},
    'max_results': 2000,
}
if year_range:
    config['query']['years'] = year_range
if doc_types:
    config['query']['doc_types'] = doc_types

# validate_config rejects a credential in the file and explains every
# rejection in words, so a mistake here is readable, not a traceback.
config = validate_config(config, 'this notebook')

CONFIG_PATH = os.path.join(OUT_DIR, 'review.json')
with open(CONFIG_PATH, 'w', encoding='utf-8') as fh:
    json.dump(config, fh, ensure_ascii=False, indent=1,
              sort_keys=True)

print('Wrote', CONFIG_PATH)
print(json.dumps(config, indent=1, sort_keys=True))

## 7. See the plan before anything is retrieved

This shows the query exactly as each database will receive it, in that
database's own syntax, and nothing is sent anywhere. Two reasons to
read it carefully:

1. These strings are what PRISMA-S Item 8 asks you to publish -- the
   full search strategies *as run*. Copy them into your methods
   section.
2. Filters some APIs cannot express are listed too. A limit that the
   API silently drops must be applied during screening instead, and
   said so in the appendix.

In [ ]:
# Step 7: dry run -- compiles the query, retrieves nothing.
from corpusslr.cli import build_query, compile_for

query = build_query(config)
for spec in config['databases']:
    compiled = compile_for(spec['name'], query)
    print('=' * 70)
    print(DATABASES[spec['name']]['label'])
    print(compiled['query'])
    if compiled.get('filters'):
        print('filters:', compiled['filters'])

for warning in dict.fromkeys(query.warnings):
    print('\nwarning:', warning)

## 8. Run the search

Now the retrieval happens. This can take several minutes: the
databases are rate-limited on purpose and CorpusSLR respects their
limits rather than hammering them.

What happens in this one step, in order: every database is queried;
missing abstracts are recovered from Crossref and PubMed; duplicates
are removed by a cascade that matches identifiers first and then title
similarity with a year tolerance, recording which rule removed which
record; the PRISMA 2020 flow diagram and the PRISMA-S appendix are
generated from the actual counts; and the export files are written.

`--keep-going` means a database without a key is skipped and recorded
as a gap in the run summary, instead of stopping the whole review.
That is deliberate: a missing subscription should not cost you the
databases you *can* reach, but it must be visible in the report.

In [ ]:
# Step 8: run the review. Progress appears below; be patient.
from corpusslr.cli import main as corpusslr_main

exit_code = corpusslr_main(['run', '-c', CONFIG_PATH,
                            '-C', OUT_DIR, '--keep-going'])

if exit_code == 0:
    print('\nSearch finished.')
else:
    print('\nThe run stopped with code {}. The lines above say what '
          'failed. The usual causes are a missing API key (step 3), '
          'a query that matched nothing (step 5), or a database '
          'being temporarily unavailable.'.format(exit_code))

## 9. What the numbers say

The run summary is the audit trail: how many records each database
contributed, how many duplicates were removed and by which rule, and
the configuration checksum that ties these numbers to this exact
search strategy. Report the identified and unique totals in your
PRISMA diagram.

In [ ]:
# Step 9: read the counts back from the run summary.
import json
import os

summary_path = os.path.join(OUT_DIR, 'run_summary.json')
if not os.path.isfile(summary_path):
    print('No run summary yet -- step 8 has not completed.')
else:
    with open(summary_path, 'r', encoding='utf-8') as fh:
        summary = json.load(fh)
    print('records identified :', summary.get('identified'))
    for source, count in sorted(
            (summary.get('identified_by_source') or {}).items()):
        print('   {:<22} {}'.format(source, count))
    dedup = summary.get('dedup') or {}
    print('duplicates removed :', dedup.get('removed'))
    for rule, count in sorted((dedup.get('by_method') or {}).items()):
        print('   {:<22} {}'.format(rule, count))
    print('unique records     :', dedup.get('after'))
    for failure in summary.get('failures') or []:
        print('not searched:', failure.get('database'), '--',
              failure.get('reason'))

## 10. The canonical export: Scopus CSV

After deduplication the corpus is written in the **Scopus CSV column
layout**. That is the format bibliometrix, VOSviewer and EmbedSLR all
read without conversion, so using it as the hand-off means no renaming
step that none of those tools document.

In [ ]:
# Step 10: write the Scopus-format CSV.
import csv
import os
from corpusslr.cli import read_corpus
from corpusslr.export import to_scopus_csv

unique_path = os.path.join(OUT_DIR, 'corpus_unique.json')
if not os.path.isfile(unique_path):
    print('No deduplicated corpus yet -- run step 8 first.')
else:
    corpus, result = read_corpus(unique_path)
    scopus_path = os.path.join(OUT_DIR, 'corpus_scopus.csv')
    to_scopus_csv(corpus.records, scopus_path)
    with open(scopus_path, newline='', encoding='utf-8') as fh:
        rows = list(csv.reader(fh))
    print('Wrote {} -- {} columns, {} records.'.format(
        scopus_path, len(rows[0]), len(rows) - 1))
    print('First columns:', ', '.join(rows[0][:6]))

## 11. The PRISMA-S appendix

PRISMA-S is the search-reporting extension to PRISMA: it asks for the
databases and platforms, the full strategies as run, the date of the
search, any limits applied and how duplicates were handled. CorpusSLR
generates the appendix from the run itself, so the appendix cannot
drift out of agreement with what was actually searched.

Read it, add the parts only you know (the protocol registration, peer
review of the strategy) and submit it as supplementary material.

In [ ]:
# Step 11: show the PRISMA-S appendix.
import os

appendix_path = os.path.join(OUT_DIR, 'prisma_s_appendix.md')
if os.path.isfile(appendix_path):
    with open(appendix_path, 'r', encoding='utf-8') as fh:
        print(fh.read())
else:
    print('No appendix yet -- run step 8 first.')

## 12. Download everything

This packs the output folder into one zip and downloads it to your
computer. Colab deletes the runtime's files when the session ends, so
**do not skip this step** -- an unsaved corpus is a search you have to
run again.

Outside Colab (Jupyter on your own machine) the download call is
skipped and the zip is simply left in the folder.

In [ ]:
# Step 12: zip and download.
import os
import shutil

archive = shutil.make_archive('corpusslr_results', 'zip', OUT_DIR)
print('Created', archive,
      '({:.1f} KB)'.format(os.path.getsize(archive) / 1024))

try:
    from google.colab import files as colab_files
except ImportError:
    print('Not running in Colab; the zip is in the folder above.')
else:
    colab_files.download(archive)

## What to do next

1. **Screen** `screening.csv` in ASReview, Rayyan or a spreadsheet,
   with two reviewers where your protocol requires it.
2. **Record your screening counts** in `review.json` under
   `screening` -- how many records you excluded, how many full texts
   you could not retrieve, how many studies you included. Rerun
   `corpusslr prisma` and the flow diagram becomes the complete PRISMA
   2020 figure rather than only its identification stage.
3. **Deposit** `review.json`, `run_summary.json`,
   `prisma_s_appendix.md` and `dedup_report.csv`. Together they let a
   reader repeat your search and recompute every number you report --
   which is the whole point.

Prefer a terminal? The same review runs as a guided menu with
`corpusslr tui`, or from a configuration file with
`corpusslr run -c review.json`.